# 💼 Job Market Analyzer — Google Colab

Analyzes job postings to identify in-demand skills, salaries, locations, and experience requirements.

**Technologies:** Python, Pandas, SQL, Data Visualization

This notebook clones the project from GitHub, loads the sample dataset (or your own CSV), and runs the same analysis functions used by the Streamlit app.

> Tip: In Colab, go to **Runtime → Run all** to execute the whole notebook top to bottom.

## 1. Setup — clone the repo and install dependencies

In [ ]:
REPO_URL = "https://github.com/krischan820/krischan820.git"
BRANCH = "main"  # change to the branch you're working on if needed
PROJECT_DIR = "job-market-analyzer"

import os

if not os.path.exists("krischan820"):
    !git clone --branch $BRANCH --single-branch $REPO_URL
else:
    !cd krischan820 && git pull

%cd krischan820/{PROJECT_DIR}
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.append(".")

import pandas as pd
import matplotlib.pyplot as plt

from analyzer import (
    load_data,
    top_skills,
    salary_by_group,
    experience_distribution,
    to_sqlite,
    run_select_query,
)

plt.rcParams["figure.figsize"] = (10, 5)

## 2. Load the data

Uses the bundled sample dataset (`data/job_postings_sample.csv`, 150 synthetic postings) by default.

To use your own CSV instead, run the optional upload cell below — it must have the same columns: `job_title, company, location, remote, experience_level, employment_type, salary_min, salary_max, skills, date_posted`.

In [ ]:
df = load_data("data/job_postings_sample.csv")
print(f"Loaded {len(df):,} job postings")
df.head()

In [ ]:
# Optional: upload your own CSV instead of the sample data.
# Uncomment and run this cell to use it.

# from google.colab import files
# uploaded = files.upload()
# filename = next(iter(uploaded))
# df = load_data(filename)
# print(f"Loaded {len(df):,} job postings from {filename}")
# df.head()

## 3. In-demand skills

In [ ]:
skills_df = top_skills(df, n=15)
display(skills_df)

plt.barh(skills_df["skill"][::-1], skills_df["postings"][::-1], color="#2E86AB")
plt.xlabel("Job postings")
plt.title("Most in-demand skills")
plt.tight_layout()
plt.show()

## 4. Salary analysis

In [ ]:
by_location = salary_by_group(df, "location").head(10)
display(by_location)

plt.barh(by_location["location"][::-1], by_location["avg_salary"][::-1], color="#2E86AB")
plt.xlabel("Average salary ($)")
plt.title("Average salary by location (top 10)")
plt.tight_layout()
plt.show()

In [ ]:
by_exp = salary_by_group(df, "experience_level")
display(by_exp)

plt.bar(by_exp["experience_level"], by_exp["avg_salary"], color="#2E86AB")
plt.ylabel("Average salary ($)")
plt.title("Average salary by experience level")
plt.tight_layout()
plt.show()

## 5. Experience level & employment type distribution

In [ ]:
exp_dist = experience_distribution(df)
plt.pie(exp_dist["postings"], labels=exp_dist["experience_level"], autopct="%1.0f%%")
plt.title("Postings by experience level")
plt.show()

## 6. SQL analysis

The dataframe is loaded into an in-memory SQLite database, so you can query it with plain SQL. See `sql/analysis_queries.sql` for more example queries.

In [ ]:
conn = to_sqlite(df)

query = """
SELECT experience_level,
       COUNT(*) AS postings,
       ROUND(AVG((salary_min + salary_max) / 2.0), 0) AS avg_salary
FROM job_postings
GROUP BY experience_level
ORDER BY avg_salary DESC;
"""

run_select_query(conn, query)

In [ ]:
# Try your own query against the `job_postings` table
my_query = "SELECT job_title, COUNT(*) AS postings FROM job_postings GROUP BY job_title ORDER BY postings DESC LIMIT 10;"
run_select_query(conn, my_query)

## Next steps

- Swap in a real job-postings dataset (e.g. scraped or from a public API) with the same column schema.
- Run the same `analyzer.py` functions in the Streamlit app for an interactive dashboard — see the project `README.md` for local run and deployment instructions.
- Extend `analyzer.py` with new metrics (e.g. skill co-occurrence, salary trend over time) — both this notebook and the app will pick them up.